# Hands-on — Ray Tasks, an Actor & Ray Train (CPU-Only)
### AI Operations (AIOps) — Module 2, Lecture 8 | ~60–75 minutes

**Referenced in:** *Lecture 8 -- Ray - Module 2.pdf*, Hands-on Exercise 2

**Objective:** use all three of Ray's core building blocks on problems that actually resemble
what you'd use them for in practice, not toy placeholders:

| Ray Primitive | What You'll Build | Why It Needs THIS Primitive |
|---|---|---|
| **Task** (`@ray.remote` function) | Distributed feature engineering across partitioned user-activity data | Each partition's work is independent and stateless — perfect for parallel tasks |
| **Actor** (`@ray.remote` class) | A live experiment tracker that many parallel training runs report results to | The runs need a SHARED, MUTABLE place to record results — tasks alone can't do this |
| **Ray Train** (`TorchTrainer`) | Data-parallel training of a small neural network on a real, learnable dataset | Demonstrates actual gradient-based learning split across multiple CPU workers |

**Everything in this notebook is CPU-only.** No GPU is required or used anywhere — every
`ScalingConfig` below is `use_gpu=False`, and every model is small enough to train in seconds
on ordinary laptop hardware.

**Deliverable:**
- Section 1: engineered features for 2,000+ users, computed via Ray tasks, with a timing comparison
- Section 2: a leaderboard of 6+ model configurations, aggregated live by a Ray actor
- Section 3: a Ray Train run showing REAL learning (decreasing loss, rising validation accuracy) across 2+ CPU workers, with per-worker progress captured by an actor

> **A note on hardware:** the wall-clock speedup you observe in Section 1 depends entirely on
> how many CPU cores your machine actually has. On a normal 4–8 core laptop you should see a
> clear speedup as partitions run truly concurrently. On a constrained single-core environment
> (some cloud sandboxes), you may see little speedup or even a slowdown from Ray's own
> scheduling overhead — this is a hardware ceiling, not a bug in your code. The programming
> model (tasks, futures, `ray.get()`) is identical either way.

## Step 0 — Setup: initialize Ray

In [ ]:
import ray

if ray.is_initialized():
    ray.shutdown()
ray.init(ignore_reinit_error=True, logging_level="ERROR", log_to_driver=False)

print("Ray initialized. Available resources:", ray.available_resources())

---
## Section 1 — Ray Tasks: Distributed Feature Engineering

Recall Module 2 Lecture 1's case study: scaling a recommendation engine's feature pipeline from
50,000 to 5 million users. Spark handled the large, mostly-tabular aggregations well — but some
feature logic is inherently **sequential per user** (it depends on the ORDER of events, with
custom reset rules) and doesn't vectorize cleanly into Spark's DataFrame operations. This is
exactly the gap Ray fills: arbitrary Python, distributed.

We'll compute an **"engagement momentum"** feature per user: a rolling score that decays over
time but RESETS whenever a user goes quiet for too long — the kind of stateful, order-dependent
logic that's awkward to express as a `groupBy().agg()`.

In [ ]:
import numpy as np

def generate_user_events(user_id, rng_seed, n_events=2000):
    """Deterministic, structured synthetic click-event data for one user.
    Each user has a hidden 'propensity' that drives realistic click behavior --
    this is NOT pure random noise; there is a genuine pattern to extract."""
    rng = np.random.default_rng(rng_seed)
    propensity = rng.uniform(0.1, 0.9)
    session_gaps_min = rng.exponential(scale=1.0 / propensity, size=n_events)
    clicked = rng.random(n_events) < propensity
    session_lengths = rng.gamma(shape=2.0, scale=30 * propensity, size=n_events)
    return session_gaps_min, clicked, session_lengths


def engineer_features_plain(user_ids, seed_offset):
    """Plain, non-distributed version -- our sequential baseline."""
    results = []
    for uid in user_ids:
        gaps, clicked, lengths = generate_user_events(uid, seed_offset + uid)
        momentum = 0.0
        momentum_trace = []
        for gap, c, length in zip(gaps, clicked, lengths):
            if gap > 5.0:          # a long gap resets engagement -- hard to vectorize
                momentum = 0.0
            momentum = 0.7 * momentum + 0.3 * (1.0 if c else 0.0) * (length / 60.0)
            momentum_trace.append(momentum)
        results.append({
            "user_id": uid,
            "click_rate": float(clicked.mean()),
            "avg_session_length": float(lengths.mean()),
            "final_engagement_momentum": float(momentum_trace[-1]),
            "peak_engagement_momentum": float(max(momentum_trace)),
        })
    return results


print("Defined the feature engineering logic. Next: wrap it as a Ray task.")

### Wrap it as a Ray task

One small change — `@ray.remote` — turns this into a distributable unit of work.

In [ ]:
@ray.remote
def engineer_features_for_partition(user_ids, seed_offset):
    return engineer_features_plain(user_ids, seed_offset)


# Partition 4,000 users into 8 shards, interleaved so each shard has similar total work
N_USERS = 4000
N_PARTITIONS = 8
user_ids = list(range(N_USERS))
partitions = [user_ids[i::N_PARTITIONS] for i in range(N_PARTITIONS)]

print(f"{N_USERS} users split into {N_PARTITIONS} partitions of ~{N_USERS // N_PARTITIONS} users each")

### Run it: sequential baseline vs. Ray tasks

In [ ]:
import time

# --- Sequential baseline: plain Python, no Ray at all ---
t0 = time.time()
for p in partitions:
    engineer_features_plain(p, seed_offset=1000)
sequential_time = time.time() - t0
print(f"Sequential (plain Python, all {N_PARTITIONS} partitions): {sequential_time:.2f}s")

# --- Ray tasks: launch all 8 partitions concurrently ---
t0 = time.time()
futures = [engineer_features_for_partition.remote(p, seed_offset=1000) for p in partitions]
print("Immediately after .remote() calls, futures look like:", futures[:2], "...")
all_results = ray.get(futures)   # blocks until every partition finishes
parallel_time = time.time() - t0

flat = [r for part in all_results for r in part]
print(f"\nRay parallel ({N_PARTITIONS} tasks): {parallel_time:.2f}s")
print(f"Speedup: {sequential_time / parallel_time:.1f}x  (on a multi-core machine, expect this to approach the number of partitions)")
print(f"\nProcessed {len(flat)} users total. Sample result:")
print(flat[0])

**Discussion:** if your speedup is well under `N_PARTITIONS`, consider: how many CPU cores does
your machine actually have (`import os; os.cpu_count()`)? Ray cannot run more tasks truly
concurrently than you have cores — requesting more parallelism than your hardware supports adds
scheduling overhead without adding real throughput.

In [ ]:
import os
print("This machine reports", os.cpu_count(), "CPU cores.")

---
## Section 2 — Ray Actors: A Live Experiment Tracker

Now a different problem shape. We're going to launch SIX independent model-training tasks (three
model types, two configurations each) in parallel — but this time, we want to know, as they
complete, which one is winning, and keep a running leaderboard.

A stateless `@ray.remote` **task** can't do this: each task runs in its own isolated process with
no memory of what any other task found. We need something that PERSISTS state across many calls
from many different tasks — exactly what a Ray **actor** is for.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import make_classification

# A shared, deterministic dataset every model configuration trains against
X, y = make_classification(n_samples=2000, n_features=15, n_informative=8, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Dataset ready: {X_train.shape[0]} train rows, {X_test.shape[0]} test rows, {X_train.shape[1]} features")

### Define the actor: one instance, shared by every task

In [ ]:
@ray.remote
class ExperimentTracker:
    """A stateful coordinator that many parallel training tasks report results to.
    This is exactly the shared-state problem a stateless task cannot solve alone."""
    def __init__(self):
        self.results = []
        self.best = None

    def report(self, run_name, model_type, params, accuracy):
        entry = {"run_name": run_name, "model_type": model_type, "params": params, "accuracy": accuracy}
        self.results.append(entry)
        if self.best is None or accuracy > self.best["accuracy"]:
            self.best = entry
        return len(self.results)   # how many results the actor has seen so far

    def get_leaderboard(self, top_k=6):
        return sorted(self.results, key=lambda r: -r["accuracy"])[:top_k]

    def get_best(self):
        return self.best

    def count(self):
        return len(self.results)


tracker = ExperimentTracker.remote()   # create ONE actor instance, reused by every task below
print("Actor created. It currently holds", ray.get(tracker.count.remote()), "results.")

### Define the training task: it calls INTO the actor when it finishes

In [ ]:
@ray.remote
def train_and_report(tracker_handle, run_name, model_type, params, X_train, y_train, X_test, y_test):
    if model_type == "random_forest":
        model = RandomForestClassifier(random_state=42, n_jobs=1, **params)
    elif model_type == "logistic_regression":
        model = LogisticRegression(max_iter=500, **params)
    else:
        model = DecisionTreeClassifier(random_state=42, **params)

    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))

    # A REMOTE METHOD CALL on the actor handle -- this is what lets many independent
    # tasks all update the SAME shared state safely.
    count_so_far = ray.get(tracker_handle.report.remote(run_name, model_type, params, acc))
    return run_name, acc, count_so_far


configs = [
    ("rf-shallow",        "random_forest",       {"n_estimators": 50, "max_depth": 4}),
    ("rf-deep",           "random_forest",       {"n_estimators": 50, "max_depth": 12}),
    ("logreg-default",    "logistic_regression", {}),
    ("logreg-l2-strong",  "logistic_regression", {"C": 0.1}),
    ("tree-shallow",      "decision_tree",       {"max_depth": 3}),
    ("tree-deep",         "decision_tree",       {"max_depth": 15}),
]

futures = [
    train_and_report.remote(tracker, name, mtype, params, X_train, y_train, X_test, y_test)
    for name, mtype, params in configs
]
completed = ray.get(futures)
for name, acc, count in completed:
    print(f"{name:20s} acc={acc:.4f}   (actor had seen {count} results at that point)")

### Read the final state back from the actor

In [ ]:
print("=== Leaderboard (via the actor) ===")
for entry in ray.get(tracker.get_leaderboard.remote()):
    print(f"  {entry['run_name']:20s} {entry['accuracy']:.4f}   {entry['params']}")

print("\n=== Best model overall ===")
print(ray.get(tracker.get_best.remote()))

print("\nTotal results the actor tracked:", ray.get(tracker.count.remote()))
assert ray.get(tracker.count.remote()) == len(configs), "The actor should have seen every run!"

**Discussion:** what would go wrong if, instead of one shared `tracker`, each call to
`train_and_report.remote()` created a NEW `ExperimentTracker.remote()` internally? Try to answer
before reading on: each task would get its own private, empty tracker, see only ITS OWN result,
and the leaderboard/best-model logic would be meaningless — this is the single most common Ray
actor mistake (see Module 2's slide deck).

---
## Section 3 — Ray Train: Data-Parallel Training on a Real, Learnable Dataset

Unlike a toy dataset of pure random numbers (where there is nothing to learn), we'll generate a
dataset with genuine structure, split it ACROSS two CPU workers (data parallelism — recall the
slide deck's data-vs-model-parallelism discussion), and confirm the model actually learns: loss
should fall, validation accuracy should rise, on EACH worker's independent shard.

In [ ]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

X_raw, y_raw = make_classification(
    n_samples=4000, n_features=20, n_informative=14, n_redundant=4,
    n_classes=2, class_sep=1.2, random_state=42,
)
X_tr, X_val, y_tr, y_val = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw)

scaler = StandardScaler().fit(X_tr)
X_tr = scaler.transform(X_tr).astype(np.float32)
X_val = scaler.transform(X_val).astype(np.float32)
y_tr = y_tr.astype(np.float32)
y_val = y_val.astype(np.float32)

print(f"Training data: {X_tr.shape}, Validation data: {X_val.shape}")
print("This IS a learnable dataset (make_classification with real class structure) --")
print("not random noise, so a falling loss below will reflect genuine learning.")

### A stateful actor tracks live progress from every worker

We reuse the SAME actor pattern from Section 2 — this time coordinating live progress reports
from multiple Ray Train workers, instead of depending on Ray Train's own internal metrics API
(whose exact interface has changed across Ray versions).

In [ ]:
@ray.remote
class TrainingMonitor:
    """Same pattern as Section 2's ExperimentTracker, now coordinating live progress
    from data-parallel Ray Train workers."""
    def __init__(self):
        self.history = []

    def report_epoch(self, world_rank, epoch, train_loss, val_accuracy, shard_size):
        self.history.append({
            "world_rank": world_rank, "epoch": epoch,
            "train_loss": train_loss, "val_accuracy": val_accuracy, "shard_size": shard_size,
        })

    def get_history(self):
        return self.history


monitor = TrainingMonitor.remote()
print("Training monitor actor created.")

### Define the model and the per-worker training loop

In [ ]:
def build_model():
    return nn.Sequential(
        nn.Linear(20, 32), nn.ReLU(),
        nn.Linear(32, 16), nn.ReLU(),
        nn.Linear(16, 1),
    )


def train_loop_per_worker(config):
    import torch
    from ray.train import get_context

    ctx = get_context()
    world_size = ctx.get_world_size()   # how many workers total
    world_rank = ctx.get_world_rank()   # THIS worker's index
    monitor = config["monitor"]

    X_all, y_all = config["X_train"], config["y_train"]
    X_val_local, y_val_local = config["X_val"], config["y_val"]

    # DATA PARALLELISM: each worker trains on a DIFFERENT slice of the data.
    # world_rank=0 gets rows 0, world_size, 2*world_size, ...
    # world_rank=1 gets rows 1, world_size+1, 2*world_size+1, ...
    shard_idx = np.arange(world_rank, len(X_all), world_size)
    X_shard = torch.tensor(X_all[shard_idx])
    y_shard = torch.tensor(y_all[shard_idx]).unsqueeze(1)
    X_val_t = torch.tensor(X_val_local)
    y_val_t = torch.tensor(y_val_local).unsqueeze(1)

    model = build_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])
    loss_fn = nn.BCEWithLogitsLoss()

    for epoch in range(config["epochs"]):
        model.train()
        optimizer.zero_grad()
        preds = model(X_shard)
        loss = loss_fn(preds, y_shard)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_preds = (torch.sigmoid(model(X_val_t)) > 0.5).float()
            val_acc = (val_preds == y_val_t).float().mean().item()

        ray.get(monitor.report_epoch.remote(world_rank, epoch, loss.item(), val_acc, len(shard_idx)))


print("Training loop defined. n_jobs/threading is irrelevant here -- PyTorch's own CPU")
print("execution is what runs each worker's forward/backward pass; Ray just runs 2+ of")
print("these worker PROCESSES concurrently, each on its own data shard.")

### Launch the data-parallel training run (CPU-only, 2 workers)

`resources_per_worker={"CPU": 0.5}` deliberately makes each worker lightweight — this model and
dataset are small enough that half a CPU per worker is plenty, and it means this exact code runs
even on a machine with as few as 1-2 CPU cores total, not just on an 8-core workstation. On real
production training jobs you would size this to match your actual per-worker compute needs.

In [ ]:
from ray.train.torch import TorchTrainer
from ray.train import ScalingConfig

trainer = TorchTrainer(
    train_loop_per_worker,
    train_loop_config={
        "lr": 1e-2, "epochs": 15, "monitor": monitor,
        "X_train": X_tr, "y_train": y_tr, "X_val": X_val, "y_val": y_val,
    },
    scaling_config=ScalingConfig(
        num_workers=2, use_gpu=False,
        resources_per_worker={"CPU": 0.5},   # lightweight workers -- runs even on a 1-2 core machine
    ),
)

result = trainer.fit()
print("Training complete.")

### Inspect what actually happened, per worker, per epoch

In [ ]:
import pandas as pd

history = ray.get(monitor.get_history.remote())
df = pd.DataFrame(history).sort_values(["world_rank", "epoch"]).reset_index(drop=True)
print(df.to_string(index=False))

print("\nFinal validation accuracy per worker:")
print(df.groupby("world_rank")["val_accuracy"].last())

first_loss = df.groupby("world_rank")["train_loss"].first()
last_loss = df.groupby("world_rank")["train_loss"].last()
print("\nLoss decreased on every worker:", (last_loss < first_loss).all())
assert (last_loss < first_loss).all(), "Expected training loss to decrease for every worker!"

**Discussion:** each worker trained on a DIFFERENT half of the data (data parallelism) yet both
reached broadly similar validation accuracy by the final epoch. In a real distributed training
setup, the workers' gradients would typically be SYNCHRONIZED after each step (so both workers
converge to the SAME model) — here, for simplicity, each worker trains an independent model on
its own shard with no gradient synchronization. In 100–150 words: what would you need to add to
make this true synchronized data-parallel training (where both workers end up with IDENTICAL
model weights), and why might that be worth the added complexity for a large dataset?

_Your answer here:_



---
## Wrap-Up: When to Use Which Ray Primitive

| If your problem is... | Use a... | Because... |
|---|---|---|
| A batch of independent, stateless work items | **Task** | Each item needs no memory of the others — fire them all, collect results |
| Work that needs shared, mutable, persistent state | **Actor** | Only an actor keeps state alive across many separate calls |
| Training one model, split across workers/data shards | **Ray Train** | Handles worker process orchestration for you; you write one training function |

**Deliverable Checklist**
- [ ] Section 1: feature engineering completed for 4,000 users across 8 Ray tasks, with a reported sequential-vs-parallel timing comparison
- [ ] Section 2: 6 model configurations trained in parallel, with a correct leaderboard read back from the actor
- [ ] Section 3: a Ray Train run with 2+ workers showing DECREASING loss and RISING validation accuracy on every worker
- [ ] Written answers to both discussion prompts

*Next: proceed to `Lecture3a_Feast_FeatureStore_Exercise.ipynb`.*

In [ ]:
ray.shutdown()